# Multithreading in Java
**Multithreading** is a process of executing multiple threads simultaneously to maximize CPU utilization.
- A **Thread** is a lightweight sub-process, the smallest unit of processing.
- Threads share a common memory area (heap), reducing resource overhead.

### Core Concepts in this Notebook:
1. **Thread Life Cycle**: New, Runnable, Running, Blocked/Waiting, Terminated.
2. **Creating Threads**: Extending `Thread` class vs. Implementing `Runnable` interface.
3. **Thread Controls**: `start()`, `run()`, `sleep()`, `join()`, and Priority.
4. **Synchronization**: Race conditions, `synchronized` methods, and `synchronized` blocks.
5. **Inter-Thread Communication**: `wait()`, `notify()`, and `notifyAll()`.


---
## 1. Thread Life Cycle & States
A thread in Java always exists in one of the following 5 states (`Thread.State`):

```
  +--------+      start()      +----------+   Scheduler   +---------+
  |  New   | ----------------> | Runnable | ------------> | Running |
  +--------+                   +----------+               +---------+
                                     ^                         |
                                     | notify() / sleep done   | wait() / sleep()
                                     |                         v
                               +------------------------------------+
                               |      Blocked / Waiting / Timed     |
                               +------------------------------------+
                                             |
                                             | run() completes
                                             v
                                      +--------------+
                                      |  Terminated  |
                                      +--------------+
```

1. **New**: Created (`new Thread()`), but `start()` is not yet called.
2. **Runnable**: Ready to run and waiting for CPU allocation.
3. **Running**: CPU is actively executing the thread's `run()` method.
4. **Blocked / Waiting**: Waiting for a lock or resource (`sleep()`, `wait()`).
5. **Terminated (Dead)**: Execution has completed.


---
## 2. Creating a Thread - Way 1: Extending `Thread` Class
- Inherit from `java.lang.Thread`.
- Override the `run()` method with the task logic.
- Call `start()` to create a new call stack and execute `run()`.


In [ ]:
class TaskOne extends Thread {
    @Override
    public void run() {
        for (int i = 1; i <= 3; i++) {
            System.out.println(Thread.currentThread().getName() + " - Step: " + i);
            try {
                Thread.sleep(500); // pause 500 ms
            } catch (InterruptedException e) {
                System.out.println(e);
            }
        }
    }
}

TaskOne t1 = new TaskOne();
t1.setName("Worker-A");

TaskOne t2 = new TaskOne();
t2.setName("Worker-B");

// start() launches the thread concurrently
t1.start();
t2.start();

// Wait for threads to complete before moving to next cell
t1.join();
t2.join();
System.out.println("Both Worker threads finished.");


---
## 3. Creating a Thread - Way 2: Implementing `Runnable` Interface (Preferred)
- Why preferred? Java does not allow multiple class inheritance; implementing `Runnable` keeps your class free to extend other classes.
- Separate the `Runnable` task definition from the `Thread` execution.
- Can also be written concisely using **Lambda Expressions**.


In [ ]:
// Step 1: Define task via Runnable interface
class DownloadTask implements Runnable {
    String fileName;

    DownloadTask(String fileName) {
        this.fileName = fileName;
    }

    @Override
    public void run() {
        System.out.println("Downloading: " + fileName + " on " + Thread.currentThread().getName());
    }
}

// Step 2: Pass Runnable to Thread instance in separate clean lines
DownloadTask task1 = new DownloadTask("document.pdf");
Thread thread1 = new Thread(task1, "DownloadThread-1");

// Way 2B: Using modern Lambda expression
Runnable task2 = () -> {
    System.out.println("Image downloaded on " + Thread.currentThread().getName());
};
Thread thread2 = new Thread(task2, "DownloadThread-2");

// Start both
thread1.start();
thread2.start();

thread1.join();
thread2.join();


---
## 4. Key Thread Methods & Priorities
- **`sleep(millis)`**: Pauses the current thread for specified milliseconds.
- **`join()`**: Causes the calling thread (e.g. main) to wait until the target thread finishes.
- **`isAlive()`**: Checks if thread is still executing.
- **`setPriority(level)`**: Thread priority ranges from `1` (`MIN_PRIORITY`) to `10` (`MAX_PRIORITY`), default is `5` (`NORM_PRIORITY`).


In [ ]:
Thread threadX = new Thread(() -> {
    System.out.println("ThreadX running with priority: " + Thread.currentThread().getPriority());
});

Thread threadY = new Thread(() -> {
    System.out.println("ThreadY running with priority: " + Thread.currentThread().getPriority());
});

threadX.setPriority(Thread.MIN_PRIORITY); // 1
threadY.setPriority(Thread.MAX_PRIORITY); // 10

threadX.start();
threadY.start();

System.out.println("Is ThreadX alive? " + threadX.isAlive());
threadX.join();
threadY.join();
System.out.println("Is ThreadX alive after join? " + threadX.isAlive());


---
## 5. Thread Synchronization
- **Race Condition**: Occurs when multiple threads attempt to read and modify shared data simultaneously, causing inconsistent state.
- **`synchronized` keyword**: Acquires an intrinsic lock (monitor) on the object. Only one thread can execute the synchronized block or method at a time.


In [ ]:
class Counter {
    private int count = 0;

    // synchronized method ensures atomic updates
    public synchronized void increment() {
        count++;
    }

    public int getCount() {
        return count;
    }
}

Counter counter = new Counter();

// Create 2 threads that increment the counter 1,000 times each
Thread worker1 = new Thread(() -> {
    for (int i = 0; i < 1000; i++) counter.increment();
});

Thread worker2 = new Thread(() -> {
    for (int i = 0; i < 1000; i++) counter.increment();
});

worker1.start();
worker2.start();

worker1.join();
worker2.join();

// Without synchronization, result would be less than 2000 due to race condition
System.out.println("Final Synchronized Counter (Expected 2000): " + counter.getCount());


---
## 6. Inter-Thread Communication (`wait`, `notify`, `notifyAll`)
Allows synchronized threads to communicate their state without busy waiting / polling:
- `wait()`: Releases the monitor lock and waits until another thread calls `notify()`.
- `notify()`: Wakes up a single waiting thread on the object's monitor.
- `notifyAll()`: Wakes up all waiting threads on the object's monitor.
> Note: Must be called inside a `synchronized` context.


In [ ]:
class BankAccount {
    private int balance = 0;

    // Consumer method: waits if balance is insufficient
    public synchronized void withdraw(int amount) {
        System.out.println("Withdrawal requested for: " + amount);
        while (this.balance < amount) {
            System.out.println("Balance is low (" + this.balance + "). Waiting for deposit...");
            try {
                wait(); // release lock and wait
            } catch (InterruptedException e) {
                System.out.println(e);
            }
        }
        this.balance -= amount;
        System.out.println("Withdrawal completed! Remaining Balance: " + this.balance);
    }

    // Producer method: deposits and notifies waiting thread
    public synchronized void deposit(int amount) {
        System.out.println("Depositing: " + amount);
        this.balance += amount;
        System.out.println("Deposit successful! Current Balance: " + this.balance);
        notify(); // wake up waiting thread
    }
}

BankAccount account = new BankAccount();

// Thread 1: tries to withdraw first
Thread withdrawThread = new Thread(() -> {
    account.withdraw(500);
}, "WithdrawThread");

// Thread 2: deposits money after a short delay
Thread depositThread = new Thread(() -> {
    try {
        Thread.sleep(1000);
    } catch (InterruptedException e) {}
    account.deposit(500);
}, "DepositThread");

withdrawThread.start();
depositThread.start();

withdrawThread.join();
depositThread.join();


---
## 7. `yield()`, `ThreadGroup`, and Concurrent Collections
- **`Thread.yield()`**: Hints to the thread scheduler that the current thread is willing to yield its current use of a processor to other threads of equal priority.
- **`ThreadGroup`**: Groups multiple threads into a single managed unit (allows interrupting or checking active counts of an entire group together).
- **`ConcurrentHashMap`**: A thread-safe, high-concurrency map in `java.util.concurrent`. Unlike `Hashtable` (which locks the entire table), it uses **bucket-level / lock-striping concurrency**, allowing multiple threads to read and write concurrently without blocking.


In [ ]:
import java.util.concurrent.ConcurrentHashMap;

// 1. ThreadGroup demonstration
ThreadGroup appGroup = new ThreadGroup("WorkerGroup");

Thread tA = new Thread(appGroup, () -> {
    System.out.println("Thread A in group running...");
    Thread.yield(); // voluntary hint to yield CPU
}, "Thread-A");

Thread tB = new Thread(appGroup, () -> {
    System.out.println("Thread B in group running...");
}, "Thread-B");

tA.start();
tB.start();
System.out.println("Active Threads in Group: " + appGroup.activeCount());

tA.join();
tB.join();

// 2. ConcurrentHashMap demonstration (Thread-safe without full-table locks)
ConcurrentHashMap<String, Integer> inventory = new ConcurrentHashMap<>();
inventory.put("Laptop", 15);
inventory.put("Mouse", 50);

// Atomic compute operation
inventory.computeIfPresent("Laptop", (k, v) -> v - 1);
System.out.println("ConcurrentHashMap stock updated: " + inventory);
